El web scrapping para nutrir el dataset para aplicar PCA

Descargamos beautifulsoup que es una biblioteca de python con la función de extraer datos de paginas web que contenga o esten en formato HTML o XML

In [1]:
pip install requests beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
import requests
from bs4 import BeautifulSoup
import re
from datetime import date
import pandas as pd
import time
import numpy as np

In [3]:
url = "https://beywatch.gg/blades/wizard-rod"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    headers=headers,
    timeout=20
)

print(response.status_code)

200


Está parte es para verificar que los datos fueron probados de manera correcta.

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

<title>Wizard Rod - Beyblade X Stats, Best Combos - BEYWATCH.GG</title>


In [5]:
texto = soup.get_text(
    " ",
    strip=True
)

print(texto[:2000])

Wizard Rod - Beyblade X Stats, Best Combos - BEYWATCH.GG BEYWATCH .GG ãã¤ã¦ã©ãã Tier List Combos Builder Parts Tournaments News About TT Hasbro Tier List Combos Builder Parts Tournaments News About TT Hasbro Wizard Rod X Stamina UX Line 35.3 g Rank #2 Where to Buy 1st Place Rate i 35.5% Win Share i 64.4% Usage i 62.0% Top Cuts 5,987 Where to Buy Recommended Wizard Rod Builds from 2,892 tournaments through Sep 17, 2026 Use Wizard Rod in builder â Most played X + + Wizard Rod 1-60H 35.2% 1st place rate Â· 2,540 top cuts 40% of Wizard Rod top cuts run this build. Use in builder Combo stats Highest 1st rate X + + Wizard Rod 1-60FB 38.9% 1st place rate Â· 342 top cuts Best 1st place rate among Wizard Rod builds with 254+ top cuts, 3.7 points above the most played build. Use in builder Combo stats Different parts S + + Wizard Rod 9-60B 36.0% 1st place rate Â· 713 top cuts Best build that runs neither 1-60 nor Hexa, for when they are missing or already used elsewhere in your deck. 

In [6]:
nombre = soup.find("h1").get_text(
    strip=True
)

print(nombre)

Wizard Rod


Usamos regex que es una secuencia de caracteres que forma un patron de busqueda para encontrar, valdiar o transformar textos, valida datos comprobando si un texto tiene forma de correo electronico. telefono o contraseña segura, igual busca patrones como encontrar palabaras, numeros especificos dentro de un documento grande y sirve igual para reemplazar texto.

Con el primer codigo extrameo el 1st place rate

In [7]:
match = re.search(
    r"1st Place Rate.*?([\d.]+)%",
    texto
)

if match:
    first_rate = float(match.group(1))
else:
    first_rate = None

print(first_rate)

35.5


con este buscaremos el win share del wizard rod en la pagina de beywatch.gg, obteniendo el porcentaje de victorias en primer lugar.

In [8]:
match = re.search(
    r"Win Share.*?([\d.]+)%",
    texto
)

win_share = (
    float(match.group(1))
    if match
    else None
)

print(win_share)

64.4


Con este buscaremos el usage

In [9]:
match = re.search(
    r"Usage.*?([\d.]+)%",
    texto
)

usage = (
    float(match.group(1))
    if match
    else None
)

print(usage)

62.0


el top cuts.

In [10]:
match = re.search(
    r"Top Cuts.*?([\d,]+)",
    texto
)

top_cuts = (
    int(match.group(1).replace(",", ""))
    if match
    else None
)

print(top_cuts)

5987


Con estó lo convertiremos en una fila

In [11]:
registro = {
    "name": nombre,
    "first_rate_pct": first_rate,
    "win_share_pct": win_share,
    "usage_pct": usage,
    "top_cuts": top_cuts,
    "source_url": url
}

registro

{'name': 'Wizard Rod',
 'first_rate_pct': 35.5,
 'win_share_pct': 64.4,
 'usage_pct': 62.0,
 'top_cuts': 5987,
 'source_url': 'https://beywatch.gg/blades/wizard-rod'}

Lo convertimos en una funcion.

In [12]:
def detectar_tipo(url):

    if "/blades/" in url:
        return "blade"

    elif "/parts/ratchet-" in url:
        return "ratchet"

    elif "/parts/bit-" in url:
        return "bit"

    return "unknown"


def obtener_estadisticas_beywatch(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=20
    )

    # Si la página devuelve error, detenemos esta consulta
    response.raise_for_status()

    # Corrige caracteres raros como Â
    response.encoding = "utf-8"

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    texto = soup.get_text(
        " ",
        strip=True
    )

    # Nombre de la pieza
    nombre = soup.find("h1").get_text(
        " ",
        strip=True
    )

    # Limpieza del nombre
    nombre = nombre.replace("\xa0", " ")
    nombre = nombre.replace("Â", "")
    nombre = " ".join(nombre.split())


    # Función interna para porcentajes
    def porcentaje(etiqueta):

        match = re.search(
            rf"{etiqueta}.*?([\d.]+)%",
            texto
        )

        if match:
            return float(match.group(1))

        return None


    # Extraer Top Cuts
    match_cuts = re.search(
        r"Top Cuts.*?([\d,]+)",
        texto
    )

    if match_cuts:

        top_cuts = int(
            match_cuts.group(1)
            .replace(",", "")
        )

    else:

        top_cuts = None


    return {

        "name": nombre,

        "part_type": detectar_tipo(url),

        "first_rate_pct":
            porcentaje("1st Place Rate"),

        "win_share_pct":
            porcentaje("Win Share"),

        "usage_pct":
            porcentaje("Usage"),

        "top_cuts": top_cuts,

        "snapshot_date":
            date.today().isoformat(),

        "source_url": url
    }

In [13]:
obtener_estadisticas_beywatch(
    "https://beywatch.gg/blades/wizard-rod"
)

{'name': 'Wizard Rod',
 'part_type': 'blade',
 'first_rate_pct': 35.5,
 'win_share_pct': 64.4,
 'usage_pct': 62.0,
 'top_cuts': 5987,
 'snapshot_date': '2026-09-20',
 'source_url': 'https://beywatch.gg/blades/wizard-rod'}

In [14]:
obtener_estadisticas_beywatch(
    "https://beywatch.gg/blades/shark-scale"
)

{'name': 'Shark Scale',
 'part_type': 'blade',
 'first_rate_pct': 35.8,
 'win_share_pct': 47.0,
 'usage_pct': 44.9,
 'top_cuts': 4335,
 'snapshot_date': '2026-09-20',
 'source_url': 'https://beywatch.gg/blades/shark-scale'}

In [15]:
obtener_estadisticas_beywatch(
    "https://beywatch.gg/parts/ratchet-1-60"
)

{'name': '1-60',
 'part_type': 'ratchet',
 'first_rate_pct': 35.1,
 'win_share_pct': 77.5,
 'usage_pct': 75.5,
 'top_cuts': 7289,
 'snapshot_date': '2026-09-20',
 'source_url': 'https://beywatch.gg/parts/ratchet-1-60'}

In [16]:
obtener_estadisticas_beywatch(
    "https://beywatch.gg/blades/shark-scale"
)

{'name': 'Shark Scale',
 'part_type': 'blade',
 'first_rate_pct': 35.8,
 'win_share_pct': 47.0,
 'usage_pct': 44.9,
 'top_cuts': 4335,
 'snapshot_date': '2026-09-20',
 'source_url': 'https://beywatch.gg/blades/shark-scale'}

convertir los datos del webscrapping en el dataset

In [17]:
urls_prueba = [
    "https://beywatch.gg/blades/shark-scale",
    "https://beywatch.gg/parts/ratchet-1-60",
    "https://beywatch.gg/parts/bit-low-rush"
]

resultados = []

for url in urls_prueba:
    datos = obtener_estadisticas_beywatch(url)
    resultados.append(datos)

In [18]:
resultados

[{'name': 'Shark Scale',
  'part_type': 'blade',
  'first_rate_pct': 35.8,
  'win_share_pct': 47.0,
  'usage_pct': 44.9,
  'top_cuts': 4335,
  'snapshot_date': '2026-09-20',
  'source_url': 'https://beywatch.gg/blades/shark-scale'},
 {'name': '1-60',
  'part_type': 'ratchet',
  'first_rate_pct': 35.1,
  'win_share_pct': 77.5,
  'usage_pct': 75.5,
  'top_cuts': 7289,
  'snapshot_date': '2026-09-20',
  'source_url': 'https://beywatch.gg/parts/ratchet-1-60'},
 {'name': 'Low Rush (LR)',
  'part_type': 'bit',
  'first_rate_pct': 35.4,
  'win_share_pct': 47.9,
  'usage_pct': 46.2,
  'top_cuts': 4465,
  'snapshot_date': '2026-09-20',
  'source_url': 'https://beywatch.gg/parts/bit-low-rush'}]

convertirlo en tabla

In [19]:
df_meta = pd.DataFrame(resultados)

display(df_meta)

,name,part_type,first_rate_pct,win_share_pct,usage_pct,top_cuts,snapshot_date,source_url
0,Shark Scale,blade,35.8,47.0,44.9,4335,2026-09-20,https://beywatch.gg/blades/shark-scale
1,1-60,ratchet,35.1,77.5,75.5,7289,2026-09-20,https://beywatch.gg/parts/ratchet-1-60
2,Low Rush (LR),bit,35.4,47.9,46.2,4465,2026-09-20,https://beywatch.gg/parts/bit-low-rush


Hacemos una prueba con 10 para saber que el scrappeo funciona.

In [20]:
urls_prueba = [
    # Blades
    "https://beywatch.gg/blades/shark-scale",
    "https://beywatch.gg/blades/wizard-rod",
    "https://beywatch.gg/blades/phoenix-wing",
    "https://beywatch.gg/blades/dran-buster",

    # Ratchets
    "https://beywatch.gg/parts/ratchet-1-60",
    "https://beywatch.gg/parts/ratchet-3-60",

    # Bits
    "https://beywatch.gg/parts/bit-low-rush",
]

enumare() nos permite saber la url, try/except evita que una mala url destruya el proceso y time.sleep() es un tiempo de espera para no colapsar la pagina.

In [22]:
resultados = []

for i, url in enumerate(urls_prueba):

    print(
        f"Procesando {i+1}/{len(urls_prueba)}:",
        url
    )

    try:
        datos = obtener_estadisticas_beywatch(url)

        resultados.append(datos)

    except Exception as e:
        print("Error:", e)

    time.sleep(1.5)

Procesando 1/7: https://beywatch.gg/blades/shark-scale
Procesando 2/7: https://beywatch.gg/blades/wizard-rod
Procesando 3/7: https://beywatch.gg/blades/phoenix-wing
Procesando 4/7: https://beywatch.gg/blades/dran-buster
Procesando 5/7: https://beywatch.gg/parts/ratchet-1-60
Procesando 6/7: https://beywatch.gg/parts/ratchet-3-60
Procesando 7/7: https://beywatch.gg/parts/bit-low-rush


In [23]:
df_meta = pd.DataFrame(resultados)

display(df_meta)

,name,part_type,first_rate_pct,win_share_pct,usage_pct,top_cuts,snapshot_date,source_url
0,Shark Scale,blade,35.8,47.0,44.9,4335,2026-09-20,https://beywatch.gg/blades/shark-scale
1,Wizard Rod,blade,35.5,64.4,62.0,5987,2026-09-20,https://beywatch.gg/blades/wizard-rod
2,Phoenix Wing,blade,33.2,28.3,29.1,2815,2026-09-20,https://beywatch.gg/blades/phoenix-wing
3,Dran Buster,blade,31.9,4.4,4.7,454,2026-09-20,https://beywatch.gg/blades/dran-buster
4,1-60,ratchet,35.1,77.5,75.5,7289,2026-09-20,https://beywatch.gg/parts/ratchet-1-60
5,3-60,ratchet,33.4,45.5,46.5,4488,2026-09-20,https://beywatch.gg/parts/ratchet-3-60
6,Low Rush (LR),bit,35.4,47.9,46.2,4465,2026-09-20,https://beywatch.gg/parts/bit-low-rush


In [24]:
print("Dimensiones:")
print(df_meta.shape)

print("\nTipos de piezas:")
print(df_meta["part_type"].value_counts())

print("\nValores nulos:")
print(df_meta.isnull().sum())

Dimensiones:
(7, 8)

Tipos de piezas:
part_type
blade      4
ratchet    2
bit        1
Name: count, dtype: int64

Valores nulos:
name              0
part_type         0
first_rate_pct    0
win_share_pct     0
usage_pct         0
top_cuts          0
snapshot_date     0
source_url        0
dtype: int64


In [25]:
df_meta.to_csv(
    "beywatch_meta_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

Ahora de manera de sacar todo.

In [33]:
# Página principal de piezas de Beywatch
url_catalogo = "https://beywatch.gg/parts"

response = requests.get(
    url_catalogo,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=20
)

response.raise_for_status()
response.encoding = "utf-8"

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

Con este codigo extraemos los enlaces de blade, rachets y bits

In [34]:
paginas_catalogo = [
    ("blade", "https://beywatch.gg/"),
    ("ratchet", "https://beywatch.gg/parts/ratchets"),
    ("bit", "https://beywatch.gg/parts/bits")
]

links = []

for tipo, url_catalogo in paginas_catalogo:

    response = requests.get(
        url_catalogo,
        headers={
            "User-Agent": "Mozilla/5.0"
        },
        timeout=20
    )

    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    for etiqueta in soup.find_all("a", href=True):

        href = etiqueta["href"]

        if (
            tipo == "blade"
            and href.startswith("/blades/")
        ):
            links.append(href)

        elif (
            tipo == "ratchet"
            and href.startswith("/parts/ratchet-")
        ):
            links.append(href)

        elif (
            tipo == "bit"
            and href.startswith("/parts/bit-")
        ):
            links.append(href)

evitamos links duplicados

In [35]:
links = sorted(set(links))

convertimos las urls completas

In [36]:
urls_beywatch = [
    "https://beywatch.gg" + link
    for link in links
]

In [38]:
print(
    "Total de piezas encontradas:",
    len(urls_beywatch)
)

for url in urls_beywatch[:20]:
    print(url)

Total de piezas encontradas: 226
https://beywatch.gg/blades/aero-pegasus
https://beywatch.gg/blades/antler
https://beywatch.gg/blades/arc
https://beywatch.gg/blades/bite-croc
https://beywatch.gg/blades/black-shell
https://beywatch.gg/blades/blast
https://beywatch.gg/blades/blitz
https://beywatch.gg/blades/brave
https://beywatch.gg/blades/brush
https://beywatch.gg/blades/bullet-griffon
https://beywatch.gg/blades/bumblebee
https://beywatch.gg/blades/captain-america
https://beywatch.gg/blades/chewbacca
https://beywatch.gg/blades/clamp-crab
https://beywatch.gg/blades/clock-mirage
https://beywatch.gg/blades/cobalt-dragoon
https://beywatch.gg/blades/cobalt-drake
https://beywatch.gg/blades/crimson-garuda
https://beywatch.gg/blades/cutter-shinobi
https://beywatch.gg/blades/dark


In [37]:
blades = [
    url for url in urls_beywatch
    if "/blades/" in url
]

ratchets = [
    url for url in urls_beywatch
    if "/parts/ratchet-" in url
]

bits = [
    url for url in urls_beywatch
    if "/parts/bit-" in url
]

print("Blades:", len(blades))
print("Ratchets:", len(ratchets))
print("Bits:", len(bits))

Blades: 135
Ratchets: 37
Bits: 54


ahora si sacamos todas las 226 piezas

In [39]:
resultados_total = []
errores = []

total = len(urls_beywatch)

for i, url in enumerate(urls_beywatch, start=1):

    print(f"[{i}/{total}] {url}")

    try:

        datos = obtener_estadisticas_beywatch(url)

        resultados_total.append(datos)

    except Exception as e:

        print("ERROR:", e)

        errores.append({
            "url": url,
            "error": str(e)
        })

    # Pausa para no hacer peticiones demasiado rápidas
    time.sleep(1)

[1/226] https://beywatch.gg/blades/aero-pegasus
[2/226] https://beywatch.gg/blades/antler
[3/226] https://beywatch.gg/blades/arc
[4/226] https://beywatch.gg/blades/bite-croc
[5/226] https://beywatch.gg/blades/black-shell
[6/226] https://beywatch.gg/blades/blast
[7/226] https://beywatch.gg/blades/blitz
[8/226] https://beywatch.gg/blades/brave
[9/226] https://beywatch.gg/blades/brush
[10/226] https://beywatch.gg/blades/bullet-griffon
[11/226] https://beywatch.gg/blades/bumblebee
[12/226] https://beywatch.gg/blades/captain-america
[13/226] https://beywatch.gg/blades/chewbacca
[14/226] https://beywatch.gg/blades/clamp-crab
[15/226] https://beywatch.gg/blades/clock-mirage
[16/226] https://beywatch.gg/blades/cobalt-dragoon
[17/226] https://beywatch.gg/blades/cobalt-drake
[18/226] https://beywatch.gg/blades/crimson-garuda
[19/226] https://beywatch.gg/blades/cutter-shinobi
[20/226] https://beywatch.gg/blades/dark
[21/226] https://beywatch.gg/blades/darth-vader
[22/226] https://beywatch.gg/blad

In [40]:
df_meta_completo = pd.DataFrame(resultados_total)

display(df_meta_completo.head(20))

,name,part_type,first_rate_pct,win_share_pct,usage_pct,top_cuts,snapshot_date,source_url
0,Aero Pegasus,blade,35.7,21.9,21.0,2028.0,2026-09-20,https://beywatch.gg/blades/aero-pegasus
1,Antler,blade,40.0,0.1,0.1,5.0,2026-09-20,https://beywatch.gg/blades/antler
2,Arc,blade,29.5,0.4,0.5,44.0,2026-09-20,https://beywatch.gg/blades/arc
3,Bite Croc,blade,37.0,0.5,0.5,46.0,2026-09-20,https://beywatch.gg/blades/bite-croc
4,Black Shell,blade,47.4,0.3,0.2,19.0,2026-09-20,https://beywatch.gg/blades/black-shell
5,Blast,blade,35.4,13.0,12.6,1214.0,2026-09-20,https://beywatch.gg/blades/blast
6,Blitz,blade,34.1,0.9,0.9,85.0,2026-09-20,https://beywatch.gg/blades/blitz
7,Brave,blade,26.9,0.9,1.1,108.0,2026-09-20,https://beywatch.gg/blades/brave
8,Brush,blade,24.4,0.3,0.4,41.0,2026-09-20,https://beywatch.gg/blades/brush
9,Bullet Griffon,blade,31.5,1.9,2.0,197.0,2026-09-20,https://beywatch.gg/blades/bullet-griffon


In [41]:
print("Dimensiones:")
print(df_meta_completo.shape)

print("\nTipos de piezas:")
print(df_meta_completo["part_type"].value_counts())

print("\nValores nulos:")
print(df_meta_completo.isnull().sum())

print("\nErrores durante scraping:")
print(len(errores))

Dimensiones:
(224, 8)

Tipos de piezas:
part_type
blade      133
bit         54
ratchet     37
Name: count, dtype: int64

Valores nulos:
name               0
part_type          0
first_rate_pct    19
win_share_pct     19
usage_pct         19
top_cuts          19
snapshot_date      0
source_url         0
dtype: int64

Errores durante scraping:
2


In [43]:
print("Errores encontrados:")

for error in errores:
    print(error)

Errores encontrados:
{'url': 'https://beywatch.gg/blades/mummy-curse', 'error': "HTTPSConnectionPool(host='beywatch.gg', port=443): Read timed out. (read timeout=20)"}
{'url': 'https://beywatch.gg/blades/orochi-cluster', 'error': "('Connection aborted.', ConnectionResetError(10054, 'Se ha forzado la interrupción de una conexión existente por el host remoto', None, 10054, None))"}


In [44]:
urls_reintento = [
    "https://beywatch.gg/blades/mummy-curse",
    "https://beywatch.gg/blades/orochi-cluster"
]

recuperados = []
errores_reintento = []

for url in urls_reintento:

    try:
        datos = obtener_estadisticas_beywatch(url)
        recuperados.append(datos)

    except Exception as e:
        errores_reintento.append({
            "url": url,
            "error": str(e)
        })

In [45]:
print("Recuperados:", len(recuperados))
print("Errores restantes:", len(errores_reintento))

recuperados

Recuperados: 2
Errores restantes: 0


[{'name': 'Mummy Curse',
  'part_type': 'blade',
  'first_rate_pct': 31.0,
  'win_share_pct': 2.0,
  'usage_pct': 2.2,
  'top_cuts': 210,
  'snapshot_date': '2026-09-20',
  'source_url': 'https://beywatch.gg/blades/mummy-curse'},
 {'name': 'Orochi Cluster',
  'part_type': 'blade',
  'first_rate_pct': 22.4,
  'win_share_pct': 0.3,
  'usage_pct': 0.5,
  'top_cuts': 49,
  'snapshot_date': '2026-09-20',
  'source_url': 'https://beywatch.gg/blades/orochi-cluster'}]

In [46]:
df_recuperados = pd.DataFrame(recuperados)

df_meta_completo = pd.concat(
    [
        df_meta_completo,
        df_recuperados
    ],
    ignore_index=True
)

In [47]:
print(df_meta_completo.shape)

print(
    df_meta_completo["part_type"]
    .value_counts()
)

(226, 8)
part_type
blade      135
bit         54
ratchet     37
Name: count, dtype: int64


In [48]:
df_meta_completo.to_csv(
    "beywatch_meta_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

In [49]:
print(df_meta_completo.isnull().sum())

name               0
part_type          0
first_rate_pct    19
win_share_pct     19
usage_pct         19
top_cuts          19
snapshot_date      0
source_url         0
dtype: int64


In [50]:
sin_meta = df_meta_completo[
    df_meta_completo["usage_pct"].isnull()
]

display(
    sin_meta[
        [
            "name",
            "part_type",
            "first_rate_pct",
            "win_share_pct",
            "usage_pct",
            "top_cuts",
            "source_url"
        ]
    ]
)

,name,part_type,first_rate_pct,win_share_pct,usage_pct,top_cuts,source_url
11,Captain America,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/captain-america
13,Clamp Crab,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/clamp-crab
22,Doctor Doom,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/doctor-doom
40,Glare Cyclops,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/glare-cyclops
43,Green Goblin,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/green-goblin
44,Grogu,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/grogu
65,Lightning L-Drago (Upper Type),blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/lightning-l-drago-u...
66,Luke Skywalker,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/luke-skywalker
67,Luster Dragoon,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/luster-dragoon
72,Mister Fantastic,blade,NaN,NaN,NaN,NaN,https://beywatch.gg/blades/mister-fantastic


In [51]:
print(
    sin_meta["part_type"]
    .value_counts()
)

part_type
blade      16
bit         2
ratchet     1
Name: count, dtype: int64


In [52]:
df_meta_clean = df_meta_completo.copy()

df_meta_clean["has_meta_data"] = (
    df_meta_clean["usage_pct"]
    .notna()
    .astype(int)
)

In [55]:
df_meta_clean["meta_status"] = np.where(
    df_meta_clean["has_meta_data"] == 1,
    "observed",
    "pending_or_insufficient"
)

In [56]:
print(
    df_meta_clean["meta_status"]
    .value_counts()
)

meta_status
observed                   207
pending_or_insufficient     19
Name: count, dtype: int64


In [57]:
df_meta_clean.to_csv(
    "beywatch_meta_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

In [58]:
def crear_lookup_key(row):

    nombre = row["name"].strip()

    # Para los bits queremos la abreviatura entre paréntesis
    if row["part_type"] == "bit":

        match = re.search(r"\(([^)]+)\)", nombre)

        if match:
            return match.group(1).strip()

    # Blade y Ratchet pueden conservar el nombre
    return nombre

In [59]:
df_meta_clean["lookup_key"] = df_meta_clean.apply(
    crear_lookup_key,
    axis=1
)

In [60]:
display(
    df_meta_clean[
        [
            "name",
            "part_type",
            "lookup_key",
            "usage_pct",
            "has_meta_data"
        ]
    ].head(30)
)

,name,part_type,lookup_key,usage_pct,has_meta_data
0,Aero Pegasus,blade,Aero Pegasus,21.0,1
1,Antler,blade,Antler,0.1,1
2,Arc,blade,Arc,0.5,1
3,Bite Croc,blade,Bite Croc,0.5,1
4,Black Shell,blade,Black Shell,0.2,1
5,Blast,blade,Blast,12.6,1
6,Blitz,blade,Blitz,0.9,1
7,Brave,blade,Brave,1.1,1
8,Brush,blade,Brush,0.4,1
9,Bullet Griffon,blade,Bullet Griffon,2.0,1


realizar el merge

In [63]:
productos = pd.read_excel(
    "beyblade_dataset_v1.xlsx",
    sheet_name="Products_Meta"
)

productos_base = productos[
    [
        "product_code",
        "product_name",
        "blade",
        "ratchet",
        "bit",
        "price_median_mxn",
        "bey_count",
        "launcher_included",
        "stadium_included"
    ]
].copy()

display(productos_base)

C:\Users\tr4nvo Bv\AppData\Roaming\Python\Python311\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\tr4nvo Bv\AppData\Roaming\Python\Python311\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


,product_code,product_name,blade,ratchet,bit,price_median_mxn,bey_count,launcher_included,stadium_included
0,BX-01,Dran Sword 3-60F,Dran Sword,3-60,F,600.000,1,1,0
1,BX-02,Hells Scythe 4-60T,Hells Scythe,4-60,T,799.000,1,1,0
2,BX-04,Knight Shield 3-80N,Knight Shield,3-80,N,799.000,1,1,0
3,BX-15,Leon Claw 5-60P,Leon Claw,5-60,P,600.000,1,1,0
4,BX-23,Phoenix Wing 9-60GF,Phoenix Wing,9-60,GF,1529.000,1,1,0
5,BX-26,Unicorn Sting 5-60GP,Unicorn Sting,5-60,GP,450.000,1,0,0
6,UX-01,Dran Buster 1-60A,Dran Buster,1-60,A,600.000,1,1,0
7,UX-03,Wizard Rod 5-70DB,Wizard Rod,5-70,DB,567.665,1,0,0


In [64]:
meta_blades = df_meta_clean[
    df_meta_clean["part_type"] == "blade"
].copy()

In [65]:
productos_blade = productos_base.merge(
    meta_blades[
        [
            "lookup_key",
            "first_rate_pct",
            "win_share_pct",
            "usage_pct",
            "top_cuts",
            "has_meta_data"
        ]
    ],
    how="left",
    left_on="blade",
    right_on="lookup_key"
)

In [66]:
productos_blade = productos_blade.rename(
    columns={
        "first_rate_pct": "blade_first_rate_pct",
        "win_share_pct": "blade_win_share_pct",
        "usage_pct": "blade_usage_pct",
        "top_cuts": "blade_top_cuts",
        "has_meta_data": "blade_has_meta"
    }
)

In [67]:
display(
    productos_blade[
        [
            "product_code",
            "product_name",
            "blade",
            "blade_usage_pct",
            "blade_top_cuts",
            "blade_has_meta"
        ]
    ]
)

,product_code,product_name,blade,blade_usage_pct,blade_top_cuts,blade_has_meta
0,BX-01,Dran Sword 3-60F,Dran Sword,1.8,174.0,1
1,BX-02,Hells Scythe 4-60T,Hells Scythe,2.9,281.0,1
2,BX-04,Knight Shield 3-80N,Knight Shield,1.1,105.0,1
3,BX-15,Leon Claw 5-60P,Leon Claw,0.2,21.0,1
4,BX-23,Phoenix Wing 9-60GF,Phoenix Wing,29.1,2815.0,1
5,BX-26,Unicorn Sting 5-60GP,Unicorn Sting,2.2,213.0,1
6,UX-01,Dran Buster 1-60A,Dran Buster,4.7,454.0,1
7,UX-03,Wizard Rod 5-70DB,Wizard Rod,62.0,5987.0,1


In [68]:
meta_ratchets = df_meta_clean[
    df_meta_clean["part_type"] == "ratchet"
].copy()

meta_bits = df_meta_clean[
    df_meta_clean["part_type"] == "bit"
].copy()

In [69]:
productos_ratchet = productos_blade.merge(
    meta_ratchets[
        [
            "lookup_key",
            "first_rate_pct",
            "win_share_pct",
            "usage_pct",
            "top_cuts",
            "has_meta_data"
        ]
    ],
    how="left",
    left_on="ratchet",
    right_on="lookup_key",
    suffixes=("", "_ratchet")
)

In [70]:
productos_ratchet = productos_ratchet.rename(
    columns={
        "first_rate_pct": "ratchet_first_rate_pct",
        "win_share_pct": "ratchet_win_share_pct",
        "usage_pct": "ratchet_usage_pct",
        "top_cuts": "ratchet_top_cuts",
        "has_meta_data": "ratchet_has_meta"
    }
)

In [71]:
display(
    productos_ratchet[
        [
            "product_code",
            "product_name",
            "ratchet",
            "ratchet_usage_pct",
            "ratchet_top_cuts",
            "ratchet_has_meta"
        ]
    ]
)

,product_code,product_name,ratchet,ratchet_usage_pct,ratchet_top_cuts,ratchet_has_meta
0,BX-01,Dran Sword 3-60F,3-60,46.5,4488.0,1
1,BX-02,Hells Scythe 4-60T,4-60,2.7,256.0,1
2,BX-04,Knight Shield 3-80N,3-80,1.6,152.0,1
3,BX-15,Leon Claw 5-60P,5-60,26.2,2532.0,1
4,BX-23,Phoenix Wing 9-60GF,9-60,67.9,6555.0,1
5,BX-26,Unicorn Sting 5-60GP,5-60,26.2,2532.0,1
6,UX-01,Dran Buster 1-60A,1-60,75.5,7289.0,1
7,UX-03,Wizard Rod 5-70DB,5-70,2.5,240.0,1


In [72]:
productos_completo = productos_ratchet.merge(
    meta_bits[
        [
            "lookup_key",
            "first_rate_pct",
            "win_share_pct",
            "usage_pct",
            "top_cuts",
            "has_meta_data"
        ]
    ],
    how="left",
    left_on="bit",
    right_on="lookup_key",
    suffixes=("", "_bit")
)

In [73]:
productos_completo = productos_completo.rename(
    columns={
        "first_rate_pct": "bit_first_rate_pct",
        "win_share_pct": "bit_win_share_pct",
        "usage_pct": "bit_usage_pct",
        "top_cuts": "bit_top_cuts",
        "has_meta_data": "bit_has_meta"
    }
)

In [74]:
display(
    productos_completo[
        [
            "product_code",
            "product_name",
            "bit",
            "bit_usage_pct",
            "bit_top_cuts",
            "bit_has_meta"
        ]
    ]
)

,product_code,product_name,bit,bit_usage_pct,bit_top_cuts,bit_has_meta
0,BX-01,Dran Sword 3-60F,F,3.2,313.0,1
1,BX-02,Hells Scythe 4-60T,T,6.2,602.0,1
2,BX-04,Knight Shield 3-80N,N,0.6,60.0,1
3,BX-15,Leon Claw 5-60P,P,12.5,1205.0,1
4,BX-23,Phoenix Wing 9-60GF,GF,0.7,69.0,1
5,BX-26,Unicorn Sting 5-60GP,GP,1.0,94.0,1
6,UX-01,Dran Buster 1-60A,A,0.7,67.0,1
7,UX-03,Wizard Rod 5-70DB,DB,0.7,64.0,1


In [75]:
productos_completo["meta_coverage"] = (
    productos_completo["blade_has_meta"]
    + productos_completo["ratchet_has_meta"]
    + productos_completo["bit_has_meta"]
) / 3

In [76]:
display(
    productos_completo[
        [
            "product_code",
            "product_name",
            "blade_has_meta",
            "ratchet_has_meta",
            "bit_has_meta",
            "meta_coverage"
        ]
    ]
)

,product_code,product_name,blade_has_meta,ratchet_has_meta,bit_has_meta,meta_coverage
0,BX-01,Dran Sword 3-60F,1,1,1,1.0
1,BX-02,Hells Scythe 4-60T,1,1,1,1.0
2,BX-04,Knight Shield 3-80N,1,1,1,1.0
3,BX-15,Leon Claw 5-60P,1,1,1,1.0
4,BX-23,Phoenix Wing 9-60GF,1,1,1,1.0
5,BX-26,Unicorn Sting 5-60GP,1,1,1,1.0
6,UX-01,Dran Buster 1-60A,1,1,1,1.0
7,UX-03,Wizard Rod 5-70DB,1,1,1,1.0


In [77]:
display(
    productos_completo[
        [
            "product_code",
            "product_name",

            "blade_usage_pct",
            "ratchet_usage_pct",
            "bit_usage_pct",

            "blade_first_rate_pct",
            "ratchet_first_rate_pct",
            "bit_first_rate_pct",

            "meta_coverage"
        ]
    ]
)

,product_code,product_name,blade_usage_pct,ratchet_usage_pct,bit_usage_pct,blade_first_rate_pct,ratchet_first_rate_pct,bit_first_rate_pct,meta_coverage
0,BX-01,Dran Sword 3-60F,1.8,46.5,3.2,27.6,33.4,23.3,1.0
1,BX-02,Hells Scythe 4-60T,2.9,2.7,6.2,32.7,27.3,30.2,1.0
2,BX-04,Knight Shield 3-80N,1.1,1.6,0.6,34.3,25.7,28.3,1.0
3,BX-15,Leon Claw 5-60P,0.2,26.2,12.5,23.8,32.5,31.5,1.0
4,BX-23,Phoenix Wing 9-60GF,29.1,67.9,0.7,33.2,33.9,27.5,1.0
5,BX-26,Unicorn Sting 5-60GP,2.2,26.2,1.0,28.6,32.5,31.9,1.0
6,UX-01,Dran Buster 1-60A,4.7,75.5,0.7,31.9,35.1,26.9,1.0
7,UX-03,Wizard Rod 5-70DB,62.0,2.5,0.7,35.5,30.4,34.4,1.0


In [78]:
productos_completo["meta_usage_mean"] = (
    productos_completo[
        [
            "blade_usage_pct",
            "ratchet_usage_pct",
            "bit_usage_pct"
        ]
    ]
    .mean(axis=1)
)

In [79]:
productos_completo["meta_usage_max"] = (
    productos_completo[
        [
            "blade_usage_pct",
            "ratchet_usage_pct",
            "bit_usage_pct"
        ]
    ]
    .max(axis=1)
)

In [80]:
display(
    productos_completo[
        [
            "product_code",
            "product_name",
            "meta_usage_mean",
            "meta_usage_max",
            "meta_coverage"
        ]
    ]
)

,product_code,product_name,meta_usage_mean,meta_usage_max,meta_coverage
0,BX-01,Dran Sword 3-60F,17.166667,46.5,1.0
1,BX-02,Hells Scythe 4-60T,3.933333,6.2,1.0
2,BX-04,Knight Shield 3-80N,1.100000,1.6,1.0
3,BX-15,Leon Claw 5-60P,12.966667,26.2,1.0
4,BX-23,Phoenix Wing 9-60GF,32.566667,67.9,1.0
5,BX-26,Unicorn Sting 5-60GP,9.800000,26.2,1.0
6,UX-01,Dran Buster 1-60A,26.966667,75.5,1.0
7,UX-03,Wizard Rod 5-70DB,21.733333,62.0,1.0


Realizar el dataset por precio

In [81]:
columnas_precio = [
    "source",              # rucua / amazon_mx / mercari_jp
    "listing_title",
    "product_code",        # BX-23, UX-03, etc. si se puede identificar
    "bey_name",
    "region",              # JP / MX / Global
    "manufacturer",        # Takara Tomy / Hasbro
    "edition",             # normal / red ver / metal coat / etc.
    "package_type",        # booster / starter / set / stadium set
    "bey_count",
    "launcher_included",
    "stadium_included",
    "price",
    "currency",
    "condition",           # new / used
    "stock_status",
    "listing_url",
    "snapshot_date"
]

primero iniciamos con rucua

In [82]:
url_rucua = "https://rucua.com/collections/vendors?q=Takara+Tomy"

response = requests.get(
    url_rucua,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

response.raise_for_status()
response.encoding = "utf-8"

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

In [83]:
links_rucua = []

for a in soup.find_all("a", href=True):

    href = a["href"]

    if href.startswith("/products/"):
        links_rucua.append(href)

links_rucua = sorted(set(links_rucua))

print(
    "Productos encontrados:",
    len(links_rucua)
)

for link in links_rucua[:20]:
    print(link)

Productos encontrados: 51
/products/bx-00-beyblade-25th-anniversary-set
/products/bx-00-bit-set-f-t-b-n-gold-x-black
/products/bx-00-bit-set-f-t-b-n-silver-x-white
/products/bx-00-booster-cobalt-drake-4-60f-clear-ver
/products/bx-00-booster-cobalt-drake-4-60f-metal-coat-blue-ver
/products/bx-00-booster-draciel-shield-7-60d
/products/bx-00-booster-dragoon-storm-4-60ra
/products/bx-00-booster-dran-sword-3-60f-version-2-0
/products/bx-00-booster-dranzer-spiral-3-80t
/products/bx-00-booster-dranzer-spiral-3-80t-black-ver
/products/bx-00-booster-drigger-slash-4-80p
/products/bx-00-booster-hells-chain-5-60ht-metal-coat-black
/products/bx-00-booster-hells-size-4-60t-metal-coat-gold
/products/bx-00-booster-leon-claw-5-60p-metal-coat-gold
/products/bx-00-booster-mammoth-task-metal-coat-black
/products/bx-00-booster-rock-leone-6-80gn
/products/bx-00-booster-shark-edge-5-60gf-metal-coat-blue
/products/bx-00-cobalt-dragoon-9-60f-metal-coat-white-j-league-ver
/products/bx-00-corocoro-croc-crunch-2-

In [84]:
urls_rucua = [
    "https://rucua.com" + link
    for link in links_rucua
]

In [88]:
url_prueba_rucua = urls_rucua[0]

print(url_prueba_rucua)

https://rucua.com/products/bx-00-beyblade-25th-anniversary-set


In [89]:
import json

scripts_json = soup.find_all(
    "script",
    type="application/ld+json"
)

print("Bloques JSON-LD encontrados:", len(scripts_json))

Bloques JSON-LD encontrados: 2


In [90]:
for i, script in enumerate(scripts_json):
    print("\n--- BLOQUE", i, "---")
    print(script.get_text(strip=True)[:2000])


--- BLOQUE 0 ---
{"@context":"http:\/\/schema.org\/","@id":"\/products\/bx-00-beyblade-25th-anniversary-set#product","@type":"Product","brand":{"@type":"Brand","name":"Takara Tomy"},"category":"Trompos de batalla","description":"BEYBLADE X es un gear sport que presenta batallas extremas con increíble velocidad e impacto gracias a su truco de superaceleración, X-Dash.\n Para conmemorar el 25 aniversario de Beyblade. El set incluye Dragoon Storm 4-60RA de \"Bakuten Shoot Beyblade\", Storm Pegasus 3-70RA de \"Metal Fight Beyblade\", Victory Valkyrie 2-60RA de \"Beyblade Burst\" y Dran Sword 3-60F Holo Sticker Ver. de \"BEYBLADE X\". \nLos Beyblades de los personajes principales de las primeras temporadas de cada serie de anime ya están disponibles en versiones BEYBLADE X.\nContenido del producto: Blade (4), Ratchet (4), Bit (4), lanzador de jareta (4), jareta (4), Beycode (1), manual de instrucciones (1).","image":"https:\/\/rucua.com\/cdn\/shop\/files\/49_e5e266ce-6c5f-4d0e-8b4d-e76fea3

In [91]:
import json

raw_producto = scripts_json[0].get_text(strip=True)

producto_json = json.loads(raw_producto)

print(type(producto_json))
print(producto_json.keys())

<class 'dict'>
dict_keys(['@context', '@id', '@type', 'brand', 'category', 'description', 'image', 'name', 'offers', 'url'])


In [92]:
print("Tipo:", producto_json.get("@type"))
print("Nombre:", producto_json.get("name"))
print("Marca:", producto_json.get("brand"))

Tipo: Product
Nombre: BX-00 Beyblade 25th Anniversary Set
Marca: {'@type': 'Brand', 'name': 'Takara Tomy'}


In [93]:
print(
    json.dumps(
        producto_json.get("offers"),
        indent=2,
        ensure_ascii=False
    )
)

{
  "@id": "/products/bx-00-beyblade-25th-anniversary-set?variant=47829099446435#offer",
  "@type": "Offer",
  "availability": "http://schema.org/OutOfStock",
  "price": "5199.00",
  "priceCurrency": "MXN",
  "url": "https://rucua.com/products/bx-00-beyblade-25th-anniversary-set?variant=47829099446435"
}


In [94]:
def obtener_producto_rucua(url):

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    scripts_json = soup.find_all(
        "script",
        type="application/ld+json"
    )

    producto_json = None

    for script in scripts_json:

        try:
            data = json.loads(
                script.get_text(strip=True)
            )

            if (
                isinstance(data, dict)
                and data.get("@type") == "Product"
            ):
                producto_json = data
                break

        except json.JSONDecodeError:
            continue

    if producto_json is None:
        return None

    offers = producto_json.get(
        "offers",
        {}
    )

    marca = producto_json.get(
        "brand",
        {}
    )

    # availability normalmente viene como:
    # http://schema.org/OutOfStock
    disponibilidad = offers.get(
        "availability"
    )

    if disponibilidad:
        disponibilidad = (
            disponibilidad
            .split("/")[-1]
        )

    return {

        "source": "Rucua",

        "product_name":
            producto_json.get("name"),

        "brand":
            marca.get("name")
            if isinstance(marca, dict)
            else marca,

        "price":
            float(offers["price"])
            if offers.get("price")
            else None,

        "currency":
            offers.get("priceCurrency"),

        "stock_status":
            disponibilidad,

        "listing_url":
            url,

        "snapshot_date":
            date.today().isoformat()
    }

In [95]:
producto_prueba = obtener_producto_rucua(
    url_prueba_rucua
)

producto_prueba

{'source': 'Rucua',
 'product_name': 'BX-00 Beyblade 25th Anniversary Set',
 'brand': 'Takara Tomy',
 'price': 5199.0,
 'currency': 'MXN',
 'stock_status': 'OutOfStock',
 'listing_url': 'https://rucua.com/products/bx-00-beyblade-25th-anniversary-set',
 'snapshot_date': '2026-09-20'}

In [96]:
url_booster = next(
    (url for url in urls_rucua if "booster" in url.lower()),
    None
)

url_starter = next(
    (url for url in urls_rucua if "starter" in url.lower()),
    None
)

url_set = next(
    (url for url in urls_rucua if "set" in url.lower()),
    None
)

print("Booster:", url_booster)
print("Starter:", url_starter)
print("Set:", url_set)

Booster: https://rucua.com/products/bx-00-booster-cobalt-drake-4-60f-clear-ver
Starter: https://rucua.com/products/bx-00-corocoro-starter-cobalt-dragoon-2-60c-metal-coat-black
Set: https://rucua.com/products/bx-00-beyblade-25th-anniversary-set


In [97]:
urls_prueba_rucua = [
    url
    for url in [
        url_booster,
        url_starter,
        url_set
    ]
    if url is not None
]

resultados_rucua_prueba = []

for url in urls_prueba_rucua:

    producto = obtener_producto_rucua(url)

    resultados_rucua_prueba.append(producto)

In [98]:
df_rucua_prueba = pd.DataFrame(
    resultados_rucua_prueba
)

display(df_rucua_prueba)

,source,product_name,brand,price,currency,stock_status,listing_url,snapshot_date
0,Rucua,BX-00 Booster Cobalt Drake 4-60F Clear Ver.,Takara Tomy,1599.0,MXN,OutOfStock,https://rucua.com/products/bx-00-booster-cobal...,2026-09-20
1,Rucua,BX-00 CoroCoro Starter Cobalt Dragoon 2-60C Me...,Takara Tomy,1999.0,MXN,InStock,https://rucua.com/products/bx-00-corocoro-star...,2026-09-20
2,Rucua,BX-00 Beyblade 25th Anniversary Set,Takara Tomy,5199.0,MXN,OutOfStock,https://rucua.com/products/bx-00-beyblade-25th...,2026-09-20


In [99]:
url_contenido = url_starter

response = requests.get(
    url_contenido,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

response.raise_for_status()
response.encoding = "utf-8"

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

In [100]:
descripcion = soup.find(
    "div",
    class_=lambda x: x and "product" in x.lower()
)

print(
    descripcion.get_text(
        " ",
        strip=True
    )[:3000]
    if descripcion
    else "No encontrada"
)

Zoom Ir al artículo 1 Ir al artículo 2 Ir al artículo 3 Ir al artículo 4 Ir al artículo 5 Ir al artículo 6 Takara Tomy BX-00 CoroCoro Starter Cobalt Dragoon 2-60C Metal Coat: Black Precio de oferta $ 1,999.00 MXN 0.0 Cantidad: Añadir a la cesta BEYBLADE X es un gear sport que presenta batallas extremas con increíble velocidad e impacto gracias a su truco de superaceleración, X-Dash. Contenido del producto: Blade (1), Ratchet (1), Bit (1), lanzador de cuerda (1), Beycode (1), manual de instrucciones (1) .


In [101]:
def extraer_contenido_rucua(texto):

    texto_lower = texto.lower()

    # Cantidad de blades = cantidad aproximada de Beyblades
    match_blade = re.search(
        r"blade\s*\((\d+)\)",
        texto_lower
    )

    bey_count = (
        int(match_blade.group(1))
        if match_blade
        else None
    )

    # Detectar launcher
    launcher_included = int(
        "lanzador" in texto_lower
        or "launcher" in texto_lower
    )

    # Detectar estadio
    stadium_included = int(
        "estadio" in texto_lower
        or "stadium" in texto_lower
    )

    return {
        "bey_count": bey_count,
        "launcher_included": launcher_included,
        "stadium_included": stadium_included
    }

In [102]:
contenido = extraer_contenido_rucua(texto)

print(contenido)

{'bey_count': 4, 'launcher_included': 1, 'stadium_included': 1}


In [103]:
def extraer_cantidad(texto, elemento):

    match = re.search(
        rf"{elemento}\s*\((\d+)\)",
        texto,
        re.IGNORECASE
    )

    return (
        int(match.group(1))
        if match
        else 0
    )

In [104]:
blade_count = extraer_cantidad(
    texto,
    "Blade"
)

ratchet_count = extraer_cantidad(
    texto,
    "Ratchet"
)

bit_count = extraer_cantidad(
    texto,
    "Bit"
)

In [105]:
contenido = extraer_contenido_rucua(
    soup.get_text(
        " ",
        strip=True
    )
)

In [114]:
def obtener_producto_rucua(url):

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    scripts_json = soup.find_all(
        "script",
        type="application/ld+json"
    )

    producto_json = None

    for script in scripts_json:

        try:
            data = json.loads(
                script.get_text(strip=True)
            )

            if (
                isinstance(data, dict)
                and data.get("@type") == "Product"
            ):
                producto_json = data
                break

        except json.JSONDecodeError:
            continue


    if producto_json is None:
        return None


    offers = producto_json.get(
        "offers",
        {}
    )

    marca = producto_json.get(
        "brand",
        {}
    )

    disponibilidad = offers.get(
        "availability"
    )

    if disponibilidad:
        disponibilidad = disponibilidad.split("/")[-1]

        descripcion_raw = producto_json.get("description", "")

        descripcion_limpia = BeautifulSoup(
    descripcion_raw,
    "html.parser"
).get_text(
    " ",
    strip=True
)
        contenido = extraer_contenido_rucua(
    descripcion_limpia
)
        
    return {
        "source": "Rucua",

        "product_name":
            producto_json.get("name"),

        "product_type":
            detectar_tipo_producto(
                producto_json.get("name", "")
            ),

        "brand":
            marca.get("name")
            if isinstance(marca, dict)
            else marca,

        "price":
            float(offers["price"])
            if offers.get("price")
            else None,

        "currency":
            offers.get("priceCurrency"),

        "stock_status":
            disponibilidad,

        "bey_count":
            contenido["bey_count"],

        "launcher_included":
            contenido["launcher_included"],

        "stadium_included":
            contenido["stadium_included"],

        "listing_url":
            url,

        "snapshot_date":
            date.today().isoformat()
    }

In [119]:
producto_prueba = obtener_producto_rucua(
    url_prueba_rucua
)

producto_prueba

{'source': 'Rucua',
 'product_name': 'BX-00 Beyblade 25th Anniversary Set',
 'product_type': 'Set',
 'brand': 'Takara Tomy',
 'price': 5199.0,
 'currency': 'MXN',
 'stock_status': 'OutOfStock',
 'bey_count': 4,
 'launcher_included': 1,
 'stadium_included': 0,
 'listing_url': 'https://rucua.com/products/bx-00-beyblade-25th-anniversary-set',
 'snapshot_date': '2026-09-20'}

In [116]:
descripcion_raw = producto_json.get(
    "description",
    ""
)

print(descripcion_raw[:3000])

BEYBLADE X es un gear sport que presenta batallas extremas con increíble velocidad e impacto gracias a su truco de superaceleración, X-Dash.
 Para conmemorar el 25 aniversario de Beyblade. El set incluye Dragoon Storm 4-60RA de "Bakuten Shoot Beyblade", Storm Pegasus 3-70RA de "Metal Fight Beyblade", Victory Valkyrie 2-60RA de "Beyblade Burst" y Dran Sword 3-60F Holo Sticker Ver. de "BEYBLADE X". 
Los Beyblades de los personajes principales de las primeras temporadas de cada serie de anime ya están disponibles en versiones BEYBLADE X.
Contenido del producto: Blade (4), Ratchet (4), Bit (4), lanzador de jareta (4), jareta (4), Beycode (1), manual de instrucciones (1).


In [117]:
def detectar_tipo_producto(nombre):

    nombre = nombre.lower()

    if "booster" in nombre:
        return "Booster"

    elif "starter" in nombre:
        return "Starter"

    elif "set" in nombre:
        return "Set"

    else:
        return "Other"

Validar la extracción del json de rucua para los datos en los otros 2 links antes del webscrapping.

In [120]:
urls_prueba_rucua = [
    url_booster,
    url_starter,
    url_set
]

resultados_prueba = []

for url in urls_prueba_rucua:

    producto = obtener_producto_rucua(url)

    resultados_prueba.append(producto)

df_prueba = pd.DataFrame(resultados_prueba)

display(
    df_prueba[
        [
            "product_name",
            "product_type",
            "price",
            "stock_status",
            "bey_count",
            "launcher_included",
            "stadium_included"
        ]
    ]
)

,product_name,product_type,price,stock_status,bey_count,launcher_included,stadium_included
0,BX-00 Booster Cobalt Drake 4-60F Clear Ver.,Booster,1599.0,OutOfStock,1,0,0
1,BX-00 CoroCoro Starter Cobalt Dragoon 2-60C Me...,Starter,1999.0,InStock,1,1,0
2,BX-00 Beyblade 25th Anniversary Set,Set,5199.0,OutOfStock,4,1,0


ahora el scrapp a la pagina de rucua

In [121]:
import time

resultados_rucua = []
errores_rucua = []

total = len(urls_rucua)

for i, url in enumerate(urls_rucua, start=1):

    print(f"[{i}/{total}] {url}")

    try:

        producto = obtener_producto_rucua(url)

        if producto is not None:
            resultados_rucua.append(producto)

        else:
            errores_rucua.append({
                "url": url,
                "error": "No se encontró Product JSON-LD"
            })

    except Exception as e:

        print("ERROR:", e)

        errores_rucua.append({
            "url": url,
            "error": str(e)
        })

    time.sleep(1)

[1/51] https://rucua.com/products/bx-00-beyblade-25th-anniversary-set
[2/51] https://rucua.com/products/bx-00-bit-set-f-t-b-n-gold-x-black
[3/51] https://rucua.com/products/bx-00-bit-set-f-t-b-n-silver-x-white
[4/51] https://rucua.com/products/bx-00-booster-cobalt-drake-4-60f-clear-ver
[5/51] https://rucua.com/products/bx-00-booster-cobalt-drake-4-60f-metal-coat-blue-ver
[6/51] https://rucua.com/products/bx-00-booster-draciel-shield-7-60d
[7/51] https://rucua.com/products/bx-00-booster-dragoon-storm-4-60ra
[8/51] https://rucua.com/products/bx-00-booster-dran-sword-3-60f-version-2-0
[9/51] https://rucua.com/products/bx-00-booster-dranzer-spiral-3-80t
[10/51] https://rucua.com/products/bx-00-booster-dranzer-spiral-3-80t-black-ver
[11/51] https://rucua.com/products/bx-00-booster-drigger-slash-4-80p
[12/51] https://rucua.com/products/bx-00-booster-hells-chain-5-60ht-metal-coat-black
[13/51] https://rucua.com/products/bx-00-booster-hells-size-4-60t-metal-coat-gold
[14/51] https://rucua.com/

In [122]:
df_rucua = pd.DataFrame(resultados_rucua)

display(df_rucua.head(20))

,source,product_name,product_type,brand,price,currency,stock_status,bey_count,launcher_included,stadium_included,listing_url,snapshot_date
0,Rucua,BX-00 Beyblade 25th Anniversary Set,Set,Takara Tomy,5199.0,MXN,OutOfStock,4.0,1,0,https://rucua.com/products/bx-00-beyblade-25th...,2026-09-20
1,Rucua,BX-00 Bit Set F/T/B/N Gold x Black,Set,Takara Tomy,499.0,MXN,InStock,NaN,0,0,https://rucua.com/products/bx-00-bit-set-f-t-b...,2026-09-20
2,Rucua,BX-00 Bit Set F/T/B/N Silver x White,Set,Takara Tomy,499.0,MXN,InStock,NaN,0,0,https://rucua.com/products/bx-00-bit-set-f-t-b...,2026-09-20
3,Rucua,BX-00 Booster Cobalt Drake 4-60F Clear Ver.,Booster,Takara Tomy,1599.0,MXN,OutOfStock,1.0,0,0,https://rucua.com/products/bx-00-booster-cobal...,2026-09-20
4,Rucua,BX-00 Booster Cobalt Drake 4-60F Metal Coat: B...,Booster,Takara Tomy,4699.0,MXN,InStock,1.0,0,0,https://rucua.com/products/bx-00-booster-cobal...,2026-09-20
5,Rucua,BX-00 Booster Draciel Shield 7-60D,Booster,Takara Tomy,599.0,MXN,OutOfStock,1.0,0,0,https://rucua.com/products/bx-00-booster-draci...,2026-09-20
6,Rucua,BX-00 Booster Dragoon Storm 4-60RA,Booster,Takara Tomy,599.0,MXN,OutOfStock,1.0,0,0,https://rucua.com/products/bx-00-booster-drago...,2026-09-20
7,Rucua,BX-00 Booster Dran Sword 3-60F Version 2.0,Booster,Takara Tomy,999.0,MXN,OutOfStock,1.0,0,0,https://rucua.com/products/bx-00-booster-dran-...,2026-09-20
8,Rucua,BX-00 Booster Dranzer Spiral 3-80T,Booster,Takara Tomy,599.0,MXN,InStock,1.0,0,0,https://rucua.com/products/bx-00-booster-dranz...,2026-09-20
9,Rucua,BX-00 Booster Dranzer Spiral 3-80T Black Ver.,Booster,Takara Tomy,699.0,MXN,OutOfStock,1.0,0,0,https://rucua.com/products/bx-00-booster-dranz...,2026-09-20


In [123]:
print("Dimensiones:")
print(df_rucua.shape)

print("\nTipos de producto:")
print(df_rucua["product_type"].value_counts())

print("\nEstado de stock:")
print(df_rucua["stock_status"].value_counts())

print("\nValores nulos:")
print(df_rucua.isnull().sum())

print("\nErrores:")
print(len(errores_rucua))

Dimensiones:
(49, 12)

Tipos de producto:
product_type
Booster    18
Other      14
Starter    12
Set         5
Name: count, dtype: int64

Estado de stock:
stock_status
InStock       29
OutOfStock    20
Name: count, dtype: int64

Valores nulos:
source               0
product_name         0
product_type         0
brand                0
price                0
currency             0
stock_status         0
bey_count            9
launcher_included    0
stadium_included     0
listing_url          0
snapshot_date        0
dtype: int64

Errores:
2


In [124]:
print("\nCantidad de Beys:")
print(df_rucua["bey_count"].value_counts(dropna=False))

print("\nIncluyen launcher:")
print(df_rucua["launcher_included"].value_counts(dropna=False))

print("\nIncluyen estadio:")
print(df_rucua["stadium_included"].value_counts(dropna=False))


Cantidad de Beys:
bey_count
1.0    38
NaN     9
4.0     1
3.0     1
Name: count, dtype: int64

Incluyen launcher:
launcher_included
0    34
1    15
Name: count, dtype: int64

Incluyen estadio:
stadium_included
0    46
1     3
Name: count, dtype: int64


In [125]:
def extraer_contenido_rucua(texto):

    def cantidad(elementos):

        for elemento in elementos:

            match = re.search(
                rf"{elemento}\s*\((\d+)\)",
                texto,
                re.IGNORECASE
            )

            if match:
                return int(match.group(1))

        return 0


    blade_count = cantidad([
        "Blade"
    ])

    ratchet_count = cantidad([
        "Ratchet"
    ])

    bit_count = cantidad([
        "Bit"
    ])

    launcher_count = cantidad([
        "Lanzador",
        "Launcher"
    ])

    stadium_count = cantidad([
        "Estadio",
        "Stadium"
    ])


    # Número de Beyblades completos que pueden formarse
    bey_count = min(
        blade_count,
        ratchet_count,
        bit_count
    )


    return {

        "blade_count":
            blade_count,

        "ratchet_count":
            ratchet_count,

        "bit_count":
            bit_count,

        "launcher_count":
            launcher_count,

        "stadium_count":
            stadium_count,

        "bey_count":
            bey_count,

        "launcher_included":
            int(launcher_count > 0),

        "stadium_included":
            int(stadium_count > 0)
    }